In [2]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

import os
import json
import yaml
from crewai import Agent, Task, Crew, LLM
from crewai_tools import SerperDevTool, ScrapeWebsiteTool, WebsiteSearchTool

In [3]:
api_key = os.environ.get('GOOGLE_API_KEY')
SERPER_API_KEY = os.environ.get('SERPER_API_KEY')

In [4]:
# Initialize the LLM here. Use a global variable to pass the key so that they are always the same
llm = LLM(
    model="gemini/gemini-1.5-flash",
    temperature=0,
    max_tokens=None,
    timeout=None,
    api_key= api_key,
)
# Warning control
warnings.filterwarnings('ignore')

In [5]:
from pydantic import BaseModel, Field
from typing import List

class SocialMediaPost(BaseModel):
    platform: str = Field(..., description="The social media platform where the post will be published (e.g., Twitter, LinkedIn).")
    content: str = Field(..., description="The content of the social media post, including any hashtags or mentions.")

class ContentOutput(BaseModel):
    article: str = Field(..., description="The article, formatted in markdown.")
    social_media_posts: List[SocialMediaPost] = Field(..., description="A list of social media posts related to the article.")

In [6]:
# Define file paths for YAML configurations
files = {
    'agents': 'config/agents.yaml',
    'tasks': 'config/tasks.yaml'
}

# Load configurations from YAML files
configs = {}
for config_type, file_path in files.items():
    with open(file_path, 'r') as file:
        configs[config_type] = yaml.safe_load(file)

# Assign loaded configurations to specific variables
agents_config = configs['agents']
tasks_config = configs['tasks']

In [7]:
tool = WebsiteSearchTool(
    config=dict(
        llm=dict(
            provider="google", # or google, openai, anthropic, llama2, ...
            config=dict(
                model="gemini-1.5-flash",
                # temperature=0.5,
                # top_p=1,
                # stream=true,
            ),
        ),
        embedder=dict(
            provider="google", # or openai, ollama, ...
            config=dict(
                model="models/embedding-001",
                task_type="retrieval_document",
                # title="Embeddings",
            ),
        ),
    )
)

In [11]:
# Creating Agents
market_news_monitor_agent = Agent(
    config=agents_config['market_news_monitor_agent'],
    tools=[SerperDevTool(), ScrapeWebsiteTool()],
    llm=llm
)

data_analyst_agent = Agent(
    config=agents_config['data_analyst_agent'],
    tools=[SerperDevTool(), tool],
    llm=llm
)

content_creator_agent = Agent(
    config=agents_config['content_creator_agent'],
    tools=[SerperDevTool(), tool],
    llm=llm
)

quality_assurance_agent = Agent(
    config=agents_config['quality_assurance_agent'],
    llm=llm
)

# Creating Tasks
monitor_financial_news_task = Task(
    config=tasks_config['monitor_financial_news'],
    agent=market_news_monitor_agent
)

analyze_market_data_task = Task(
    config=tasks_config['analyze_market_data'],
    agent=data_analyst_agent
)

create_content_task = Task(
    config=tasks_config['create_content'],
    agent=content_creator_agent,
    context=[monitor_financial_news_task, analyze_market_data_task]
)

quality_assurance_task = Task(
    config=tasks_config['quality_assurance'],
    agent=quality_assurance_agent,
    output_pydantic=ContentOutput
)

# Creating Crew
content_creation_crew = Crew(
    agents=[
        market_news_monitor_agent,
        data_analyst_agent,
        content_creator_agent,
        quality_assurance_agent
    ],
    tasks=[
        monitor_financial_news_task,
        analyze_market_data_task,
        create_content_task,
        quality_assurance_task
    ],
    verbose=True
)

2025-02-24 16:40:41,333 - 11200 - __init__.py-__init__:537 - WARNING: Overriding of current TracerProvider is not allowed


In [14]:
result = content_creation_crew.kickoff(inputs={
  'subject': 'Inflation in the Pakistan and the impact on the stock market in 2025'
})

# Agent: Lead Market Analyst
## Task: Monitor and analyze the latest news and updates related to the financial markets, with a particular focus on Inflation in the Pakistan and the impact on the stock market in 2025. Identify and summarize the most relevant and impactful news items that could influence market trends or investor decisions. Utilize financial news APIs and real-time market data tools to gather up-to-date information. Focus on detecting trends, regulatory changes, or significant economic indicators that directly relate to Inflation in the Pakistan and the impact on the stock market in 2025.



# Agent: Lead Market Analyst
## Using tool: Search the internet
## Tool Input: 
"{\"search_query\": \"Pakistan inflation 2025 stock market impact\"}"
## Tool Output: 

Search results: Title: Pakistan Jan consumer inflation eases to 9-year low | Reuters
Link: https://www.reuters.com/markets/asia/pakistan-jan-consumer-inflation-eases-9-year-low-2025-02-03/
Snippet: Pakistan's consumer 


nserting batches in chromadb: 100%|█████████████████████████████████████████████████████| 1/1 [00:01<00:00,  1.86s/it]



# Agent: Creative Content Director
## Using tool: Search in a specific website
## Tool Input: 
"{\"search_query\": \"Pakistan stock market infographic\", \"website\": \"https://www.google.com/images\"}"
## Tool Output: 
Relevant Content:
Google Images Sign inimages PakistanAboutAdvertisingBusiness How Search works PrivacyTermsSettingsSearch settingsAdvanced searchSearch historySearch helpSend feedbackDark theme: OffGoogle apps

the firms we regulate directly, and in the financial market broadly. Proposals for Comment Find, review, and submit comments on Board proposals. Select Statistical Releases Principal Economic Indicators DDP for Consumer Credit - G.19 Consumer Credit - G.19 DDP for Industrial Production and Capacity Utilization - G.17 Industrial Production and Capacity Utilization - G.17 DDP for Factors Affecting Reserve Balances - H.4.1 Factors Affecting Reserve Balances - H.4.1 DDP for Money Stock Measures - H.6 Money Stock Measures - H.6 DDP for Commercial Paper Commercial P


nserting batches in chromadb: 100%|█████████████████████████████████████████████████████| 1/1 [00:00<00:00,  2.47it/s]



# Agent: Creative Content Director
## Using tool: Search in a specific website
## Tool Input: 
"{\"search_query\": \"Pakistan inflation and KSE-100 index charts 2024-2025\", \"website\": \"https://tradingeconomics.com\"}"
## Tool Output: 
Relevant Content:
Google Images Sign inimages PakistanAboutAdvertisingBusiness How Search works PrivacyTermsSettingsSearch settingsAdvanced searchSearch historySearch helpSend feedbackDark theme: OffGoogle apps

TRADING ECONOMICS | 20 million INDICATORS FROM 196 COUNTRIES ﻿

the firms we regulate directly, and in the financial market broadly. Proposals for Comment Find, review, and submit comments on Board proposals. Select Statistical Releases Principal Economic Indicators DDP for Consumer Credit - G.19 Consumer Credit - G.19 DDP for Industrial Production and Capacity Utilization - G.17 Industrial Production and Capacity Utilization - G.17 DDP for Factors Affecting Reserve Balances - H.4.1 Factors Affecting Reserve Balances - H.4.1 DDP for Money St

In [15]:
import textwrap

posts = result.pydantic.dict()['social_media_posts']
for post in posts:
    platform = post['platform']
    content = post['content']
    print(platform)
    wrapped_content = textwrap.fill(content, width=50)
    print(wrapped_content)
    print('-' * 50)

Twitter
Pakistan's inflation plummets to a nine-year low!
What does this mean for the KSE-100 in 2025?  Find
out in our latest blog post: [link to blog post]
#PakistanEconomy #Inflation #StockMarket #KSE100
--------------------------------------------------
LinkedIn
Analyzing the impact of Pakistan's inflation
rollercoaster on the 2025 stock market. Our latest
insights offer a balanced perspective on the
opportunities and risks. [link to blog post]
#PakistanFinance #Investment #EconomicAnalysis
--------------------------------------------------
Facebook
Pakistan's inflation story is one of dramatic
change. Learn how this affects your investment
strategy in 2025. Read our insightful blog post:
[link to blog post] [Include an image or short
video summarizing key findings].
--------------------------------------------------


In [16]:
from IPython.display import display, Markdown
display(Markdown(result.pydantic.dict()['article']))

# Pakistan's Inflation Rollercoaster: Navigating the 2025 Stock Market

**Introduction:**

2024 presented Pakistan with a year of dramatic economic fluctuations, characterized by soaring inflation reaching almost 40%.  However, a remarkable turnaround in early 2025 saw inflation plummet to a nine-year low of 2.4% in January. This dramatic shift has profoundly impacted the Pakistani stock market, creating both opportunities and challenges for investors. This blog post delves into the inflation trajectory, its effects on the KSE-100 index, and provides insights for navigating the 2025 market.

**Inflation's Wild Ride:**

The initial surge in inflation during 2024 fueled uncertainty and market volatility.  However, a confluence of factors, including currency stabilization, reduced food and energy prices, and the positive impact of a $7 billion IMF facility, contributed to a significant decline. While consumer inflation has decreased dramatically, core inflation remains a concern. The State Bank of Pakistan forecasts full-year inflation (year to June 2025) to average between 5.5% and 7.5%.

[Insert chart/graph here showing inflation rate from early 2024 to projected June 2025.  Ideally, this would be an interactive chart allowing users to zoom and explore the data.]

The chart above illustrates the volatility of inflation in Pakistan.  The sharp decline from the peak in 2024 is noteworthy, but the projected core inflation remains a key factor to consider for investors.  Understanding this fluctuation is crucial for making informed investment decisions.

**Market Reaction and Forecasts:**

The initial inflationary pressures negatively impacted investor confidence, resulting in market volatility throughout 2024.  The sharp drop in inflation in early 2025, however, has significantly boosted investor sentiment. Brokerage firms are projecting substantial growth in the KSE-100 index for 2025, with forecasts ranging from a 27% increase to a remarkable 55.5% increase by year-end. Some analysts even predict the index reaching 165,000 points.

[Insert chart/graph here showing KSE-100 index performance alongside inflation rate.  Again, interactivity would enhance user experience.]

The correlation between inflation and KSE-100 performance is clearly visible in the chart.  While the initial inflation spike negatively impacted the index, the subsequent decline has fueled a significant rebound.  However, investors should remain aware of the potential for future volatility.

**Navigating the Market:**

Despite the positive outlook, it's crucial to remember that these are projections, and market conditions are inherently unpredictable. The persistence of core inflation and potential political or economic instability represent significant risks. Investors should adopt a cautious yet optimistic approach, diversifying their portfolios and closely monitoring economic indicators. The continued support of the IMF and the government's economic policies will play a vital role in sustaining the positive trajectory.

Careful portfolio diversification is key to mitigating risk.  Investors should consider spreading their investments across different sectors and asset classes to reduce exposure to any single factor.  Regular monitoring of economic indicators will allow for timely adjustments to investment strategies.

**Conclusion:**

Pakistan's economic recovery, driven by a substantial drop in inflation, presents a promising outlook for the stock market in 2025.  However, investors must remain vigilant, acknowledging the remaining risks and conducting thorough due diligence before making any investment decisions.  The information provided here is for informational purposes only and does not constitute financial advice.
